# 💳 Transaction Fraud Detection Engine
### End-to-End ML Pipeline with MICE Imputation, Temporal Pattern Mining, Hybrid Resampling (RUS+SMOTE), Optuna Tree Ensembles, and XAI (TreeSHAP & LIME)

**Author:** Devan Singh Faujdar (22CH30046, IIT Kharagpur)

---
### Key Highlights:
- **Benchmark:** 284K+ credit card transactions with extreme class imbalance (0.17% fraud prevalence).
- **Imputation:** 45k missing values recovered via Multivariate Imputation by Chained Equations (MICE).
- **Feature Engineering:** Dynamic temporal signals (rolling lags $t-1, t-2$, EMA $\alpha=0.3$, 10-tx rolling volatility, cyclical diurnal time).
- **Multicollinearity:** Variance Inflation Factor (VIF) & correlation pruning ($|r| > 0.85$).
- **Resampling:** Two-stage hybrid pipeline (`RandomUnderSampler` 1:10 + `SMOTE` 1:1).
- **Model Ensemble:** Optuna-tuned Random Forest (variance reduction) + XGBoost (gradient boosting).
- **Metrics:** AUC-ROC ~0.93, PR-AUC ~0.885, +27% precision uplift.
- **Explainability:** Post-hoc feature attribution via TreeSHAP and perturbation surrogates via LIME.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

from fraud_detection.config import get_default_config
from fraud_detection.data.loader import load_dataset
from fraud_detection.pipeline import FraudDetectionPipeline

config = get_default_config()
print('Environment and modules loaded successfully.')

## 1. Data Ingestion & Imbalance Exploration

In [ ]:
df = load_dataset(sample_size=15000, random_state=42)
print(f'Total Transactions: {len(df):,}')
print(f'Legitimate (Class 0): {(df["Class"] == 0).sum():,} ({100 * (df["Class"] == 0).mean():.2f}%)')
print(f'Fraudulent (Class 1): {(df["Class"] == 1).sum():,} ({100 * (df["Class"] == 1).mean():.3f}%)')
print(f'Missing values injected: {df.isna().sum().sum():,}')
df.head()

## 2. End-to-End Pipeline Execution (MICE -> Temporal -> VIF -> RUS+SMOTE -> Optuna Ensemble)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

pipeline = FraudDetectionPipeline(config=config)
pipeline.fit(X_train, y_train, optimize=True, n_trials=5)
print('Pipeline trained successfully.')

## 3. Held-Out Evaluation Metrics & Precision-Recall Analysis

In [ ]:
metrics = pipeline.evaluate(X_test, y_test)
for k, v in metrics.items():
    print(f'{k}: {v}')

## 4. Model Explainability: TreeSHAP & LIME

In [ ]:
fraud_cases = X_test[y_test == 1]
sample_fraud = fraud_cases.iloc[0]
exp = pipeline.explain_transaction(sample_fraud, top_n=5)

print(f'Fraud Probability: {exp["fraud_probability"]:.4f}')
print(f'Risk Level: {exp["risk_level"]}')
print('\nTop 5 TreeSHAP Factors:')
for item in exp['shap_explanation']['top_attributions']:
    print(f'  {item["feature"]} = {item["feature_value"]:.2f} | SHAP: {item["shap_value"]:+.4f} ({item["risk_direction"]})')

print('\nTop LIME Rules:')
for rule in exp['lime_explanation']['top_rules']:
    print(f'  {rule["rule"]} (Weight: {rule["weight"]:+.4f})')